# Phase 5, Module 8: Image Segmentation (Medical & Precision)
**Topic:** Pixel-Level Classification with U-Net

Segmentation is a "dense" prediction task. Instead of outputting a single number (class) or four numbers (bounding box), we output an entire image where each pixel represents a class. 

The most influential architecture here is **U-Net**. It is named for its U-shape, featuring an **Encoder** to understand "what" is in the image and a **Decoder** to determine "where" it is precisely.

### Key Math: Dice Coefficient
To measure mask quality, we use the Dice Coefficient, which measures the overlap between two sets:
$$Dice = \frac{2 |A \cap B|}{|A| + |B|}$$
Where $A$ is the predicted mask and $B$ is the ground truth.

In [1]:
import torch
import torch.nn as nn

def conv_block(in_c, out_c):
    """A standard double-convolution block used in U-Net[cite: 1]."""
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
        nn.ReLU(inplace=True)
    )

class SimpleUNet(nn.Module):
    def __init__(self):
        super(SimpleUNet, self).__init__()
        # Encoder (Downsampling path)[cite: 1]
        self.enc1 = conv_block(1, 64)
        self.pool = nn.MaxPool2d(2)
        
        # Decoder (Upsampling path)[cite: 1]
        self.up = nn.ConvTranspose2d(64, 64, kernel_size=2, stride=2)
        self.dec1 = conv_block(64, 1) # Final output mask

    def forward(self, x):
        # This is a simplified U-Net structure[cite: 1]
        x1 = self.enc1(x)
        p1 = self.pool(x1)
        u1 = self.up(p1)
        mask = self.dec1(u1)
        return mask

model = SimpleUNet()
print("Simplified U-Net model structure initialized.")

Simplified U-Net model structure initialized.


## Loss Functions for Segmentation: Dice Loss
Standard Cross-Entropy often fails in medical imaging because the "target" (like a small tumor) is tiny compared to the background. **Dice Loss** focuses purely on the overlap, ensuring the model focuses on the region of interest.

In [2]:
import torch.nn.functional as F

def dice_loss(pred, target, smooth=1.):
    """Implementation of Dice Loss for binary segmentation[cite: 1]."""
    pred = torch.sigmoid(pred)
    
    # Flatten tensors to calculate overlap[cite: 1]
    pred = pred.view(-1)
    target = target.view(-1)
    
    intersection = (pred * target).sum()
    dice = (2. * intersection + smooth) / (pred.sum() + target.sum() + smooth)
    
    return 1 - dice

# Dummy test with a simulated 10x10 image[cite: 1]
pred_mask = torch.randn(1, 1, 10, 10)
true_mask = torch.zeros(1, 1, 10, 10)
true_mask[:, :, 2:5, 2:5] = 1.0 # Simulating a small target area

loss = dice_loss(pred_mask, true_mask)
print(f"Calculated Dice Loss: {loss.item():.4f}")

Calculated Dice Loss: 0.8678


---
## Student Task: The "Surgeon's Assistant" (Segmentation)

**Topic:** Module 8 (Image Segmentation)[cite: 1]

**Goal:** Isolate specific regions (e.g., lungs or a tumor) in a 2D medical slice[cite: 1].

**Requirements:**
1. Build or use a U-Net architecture that includes **Skip Connections** (concatenating encoder features to the decoder)[cite: 1].
2. Use **Dice Loss** as the optimization objective instead of standard accuracy[cite: 1].
3. Train the model on a segmentation dataset[cite: 1].

**Deliverable:**
* A script that generates a "Predicted Mask" image overlaid on the original medical scan[cite: 1].

In [3]:
# TASK: Implement the Skip Connection logic in the U-Net forward pass[cite: 1]
# Hint: In the forward function, you should concatenate the encoder feature map 
# with the upsampled decoder feature map using torch.cat([up, skip], dim=1)

def forward_with_skips(self, x):
    # YOUR CODE HERE
    # 1. Store encoder output
    # 2. Perform pooling and upsampling
    # 3. Concatenate (Skip Connection)
    # 4. Final convolution
    pass

print("Student Task: Ready for implementation of Skip Connections.")

Student Task: Ready for implementation of Skip Connections.
